# Overlap Group-Rep Retrieval Metrics

This notebook starts from the overlap-filtered `.h5ad` files produced by `scripts/build_overlap_filtered_h5ads.py`, uses the same matched-context reconstruction as `notebooks/overlap_group_rep_signature_similarity.ipynb`, and evaluates cross-dataset retrieval rather than direct matched-sample correlation.

For each dataset pair and matched `cell_type + pert_time_h` stratum, it constructs a retrieval task between all retained non-control compound-dose conditions in that stratum.

Eligibility rules:
- each side must have at least `2` unique compounds in the stratum
- the target-side candidate pool must contain at least `5` conditions
- a query is scored only if it has at least one positive in the target pool under the chosen retrieval variant

Representations:
- signed significance: `-log10(adj.P.Value) * sign(logFC)`
- moderated `t`
- `logFC`

Similarity:
- current retrieval uses negative Euclidean distance, i.e. similarity `S(i,j) = -||r_i - r_j||_2`
- it is not using cosine similarity

Primary retrieval variant:
- **compound across doses**: positives are all target-side conditions with the same compound, regardless of dose

Sensitivity variants:
- **dose-aware compound retrieval**: positives are same-compound candidates within `|Δ log10 dose| <= 1`, with nearest-dose fallback if none exist
- **strict matched-condition retrieval**: positives are only the reciprocal-nearest-dose matches reconstructed by the overlap matching logic

For each query, the notebook reports:
- normalized best-positive rank
- Recall@1
- AUROC over positives vs wrong-compound negatives

Retrieval is done in both directions (`A -> B` and `B -> A`). Summaries are aggregated by:
1. query
2. `dataset pair + direction + cell_type + time` stratum
3. dataset-pair direction
4. symmetric dataset pair, averaging the two directions


In [52]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
import itertools

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import rankdata


sns.set_theme(style="whitegrid")


In [53]:
DATASET_ORDER = [
    "l1000_phase1",
    "l1000_phase2",
    "sciplex",
    "tahoe",
]
DISPLAY_LABELS = {
    "l1000_phase1": "L1000 Phase I",
    "l1000_phase2": "L1000 Phase II",
    "sciplex": "sci-Plex",
    "tahoe": "Tahoe-100M",
}
SOURCE_DATASET_DIRS = {
    "sciplex": Path("/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/sciplex/deg_data/group_rep/full/qc_false/filter_min_cells_10/results"),
    "tahoe": Path("/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/tahoe/deg_data/group_rep/full/qc_false/filter_min_cells_50/results"),
    "l1000_phase1": Path("/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase1/deg_data/group_rep/full/qc_false/filter_min_cells_0/results"),
    "l1000_phase2": Path("/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results"),
}
MAX_LOG10_DOSE_DIFF = 1.0
MIN_CONTEXT_SHARED_DRUGS = 10
TOP_K = 50
NUMERIC_SIG_FIGS = 12


def find_repo_root(start=None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "results" / "overlap_filtered_h5ads").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
OVERLAP_DIR = REPO_ROOT / "results" / "overlap_filtered_h5ads"
OUTPUT_DIR = REPO_ROOT / "results" / "overlap_group_rep_retrieval_metrics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Overlap directory: {OVERLAP_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Repository root: /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis
Overlap directory: /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_filtered_h5ads
Output directory: /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_group_rep_retrieval_metrics


In [54]:
def pretty_label(dataset_name: str) -> str:
    return DISPLAY_LABELS.get(dataset_name, dataset_name)


def format_numeric(value: float) -> str:
    formatted = f"{value:.{NUMERIC_SIG_FIGS}g}"
    return "0" if formatted == "-0" else formatted


def coerce_control_mask(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype("string").fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"true", "1", "yes"})


EMPTY_OVERLAP_FRAME = pd.DataFrame(
    columns=[
        "dataset_name",
        "obs_id",
        "plate",
        "well",
        "pubchem_cid",
        "cell_type",
        "pert_time_h",
        "pert_dose_uM",
        "time_key",
        "dose_key",
        "log10_dose",
    ]
)


def load_overlap_obs(dataset_name: str, overlap_dir: Path = OVERLAP_DIR) -> pd.DataFrame:
    h5ad_path = overlap_dir / f"{dataset_name}_overlap_filtered.h5ad"
    if not h5ad_path.exists():
        raise FileNotFoundError(f"Missing overlap file: {h5ad_path}")

    adata = ad.read_h5ad(h5ad_path, backed="r")
    try:
        obs = adata.obs.copy()
    finally:
        adata.file.close()

    if obs.empty:
        return EMPTY_OVERLAP_FRAME.copy()
    if "is_control" not in obs.columns:
        raise KeyError(f"{h5ad_path} is missing obs['is_control']")

    obs = obs.loc[~coerce_control_mask(obs["is_control"])].copy()
    if obs.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame = pd.DataFrame(index=obs.index.copy())
    frame["dataset_name"] = dataset_name
    frame["obs_id"] = frame.index.astype(str)
    frame["plate"] = (
        obs["plate"].astype("string").fillna("").astype(str).str.strip()
        if "plate" in obs.columns
        else ""
    )
    frame["well"] = (
        obs["well"].astype("string").fillna("").astype(str).str.strip()
        if "well" in obs.columns
        else ""
    )
    frame["pubchem_cid"] = obs["pubchem_cid"].astype("string").fillna("").astype(str).str.strip()
    frame["cell_type"] = obs["cell_type"].astype("string").fillna("").astype(str).str.strip()
    frame["pert_time_h"] = pd.to_numeric(obs["pert_time_h"], errors="coerce")
    frame["pert_dose_uM"] = pd.to_numeric(obs["pert_dose_uM"], errors="coerce")

    valid_mask = (
        (frame["pubchem_cid"] != "")
        & (frame["cell_type"] != "")
        & np.isfinite(frame["pert_time_h"].to_numpy(dtype=float))
        & np.isfinite(frame["pert_dose_uM"].to_numpy(dtype=float))
        & (frame["pert_dose_uM"].to_numpy(dtype=float) > 0.0)
    )
    frame = frame.loc[valid_mask].copy().reset_index(drop=True)
    if frame.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame["time_key"] = frame["pert_time_h"].map(lambda value: format_numeric(float(value)))
    frame["dose_key"] = frame["pert_dose_uM"].map(lambda value: format_numeric(float(value)))
    frame["log10_dose"] = np.log10(frame["pert_dose_uM"].to_numpy(dtype=np.float64))
    return frame[EMPTY_OVERLAP_FRAME.columns.tolist()]


def ensure_overlap_frame_schema(frame: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    if frame is None or frame.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame = frame.copy()
    if "dataset_name" not in frame.columns:
        frame["dataset_name"] = dataset_name
    else:
        frame["dataset_name"] = frame["dataset_name"].astype("string").fillna(dataset_name).astype(str).str.strip()

    if "obs_id" not in frame.columns:
        frame["obs_id"] = pd.Index(frame.index).astype(str)
    else:
        frame["obs_id"] = frame["obs_id"].astype("string").fillna("").astype(str).str.strip()

    for column_name in ["plate", "well"]:
        if column_name not in frame.columns:
            frame[column_name] = ""
        else:
            frame[column_name] = frame[column_name].astype("string").fillna("").astype(str).str.strip()

    required_columns = ["pubchem_cid", "cell_type", "pert_time_h", "pert_dose_uM"]
    missing_columns = [column_name for column_name in required_columns if column_name not in frame.columns]
    if missing_columns:
        raise KeyError(
            f"Overlap frame for {dataset_name} is missing required columns: {missing_columns}"
        )

    frame["pubchem_cid"] = frame["pubchem_cid"].astype("string").fillna("").astype(str).str.strip()
    frame["cell_type"] = frame["cell_type"].astype("string").fillna("").astype(str).str.strip()
    frame["pert_time_h"] = pd.to_numeric(frame["pert_time_h"], errors="coerce")
    frame["pert_dose_uM"] = pd.to_numeric(frame["pert_dose_uM"], errors="coerce")

    valid_mask = (
        (frame["pubchem_cid"] != "")
        & (frame["cell_type"] != "")
        & np.isfinite(frame["pert_time_h"].to_numpy(dtype=float))
        & np.isfinite(frame["pert_dose_uM"].to_numpy(dtype=float))
        & (frame["pert_dose_uM"].to_numpy(dtype=float) > 0.0)
    )
    frame = frame.loc[valid_mask].copy().reset_index(drop=True)
    if frame.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame["time_key"] = frame["pert_time_h"].map(lambda value: format_numeric(float(value)))
    frame["dose_key"] = frame["pert_dose_uM"].map(lambda value: format_numeric(float(value)))
    frame["log10_dose"] = np.log10(frame["pert_dose_uM"].to_numpy(dtype=np.float64))
    return frame[EMPTY_OVERLAP_FRAME.columns.tolist()]


def build_groups(frame: pd.DataFrame) -> dict[tuple[str, str, str], np.ndarray]:
    if frame.empty:
        return {}
    grouped = frame.groupby(["pubchem_cid", "cell_type", "time_key"], sort=False).groups
    return {
        key: np.asarray(list(row_positions), dtype=np.int64)
        for key, row_positions in grouped.items()
    }


def build_dataset_index(dataset_name: str) -> dict[str, object]:
    frame = ensure_overlap_frame_schema(load_overlap_obs(dataset_name), dataset_name)
    return {
        "frame": frame,
        "groups": build_groups(frame),
    }


def active_dataset_names(dataset_indices: dict[str, dict[str, object]]) -> list[str]:
    return [
        dataset_name
        for dataset_name in DATASET_ORDER
        if dataset_name in dataset_indices and not dataset_indices[dataset_name]["frame"].empty
    ]


def mutual_nearest_logdose_pairs(
    left_log10_doses: np.ndarray,
    right_log10_doses: np.ndarray,
    max_log10_dose_diff: float = MAX_LOG10_DOSE_DIFF,
) -> np.ndarray:
    if left_log10_doses.size == 0 or right_log10_doses.size == 0:
        return np.empty((0, 2), dtype=np.int64)

    diff = np.abs(left_log10_doses[:, None] - right_log10_doses[None, :])
    left_min = diff.min(axis=1, keepdims=True)
    right_min = diff.min(axis=0, keepdims=True)
    is_mnn = (
        (diff <= max_log10_dose_diff + 1e-12)
        & np.isclose(diff, left_min, rtol=0.0, atol=1e-12)
        & np.isclose(diff, right_min, rtol=0.0, atol=1e-12)
    )
    return np.argwhere(is_mnn)


MATCH_PAIR_COLUMNS = [
    "dataset_a",
    "dataset_b",
    "cell_type",
    "pubchem_cid",
    "time_key",
    "left_obs_id",
    "right_obs_id",
    "left_plate",
    "right_plate",
    "left_well",
    "right_well",
    "left_dose_key",
    "right_dose_key",
    "left_log10_dose",
    "right_log10_dose",
    "abs_delta_log10_dose",
    "matched_condition_key",
    "n_context_matching_drugs",
]


def pair_match_frame(
    left_dataset: str,
    right_dataset: str,
    left_index: dict[str, object],
    right_index: dict[str, object],
    *,
    max_log10_dose_diff: float = MAX_LOG10_DOSE_DIFF,
    min_context_shared_drugs: int = MIN_CONTEXT_SHARED_DRUGS,
) -> pd.DataFrame:
    left_frame = ensure_overlap_frame_schema(left_index["frame"], left_dataset)
    right_frame = ensure_overlap_frame_schema(right_index["frame"], right_dataset)
    left_groups = left_index["groups"]
    right_groups = right_index["groups"]

    if set(build_groups(left_frame)) != set(left_groups):
        left_groups = build_groups(left_frame)
    if set(build_groups(right_frame)) != set(right_groups):
        right_groups = build_groups(right_frame)

    if left_frame.empty or right_frame.empty:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    shared_keys = sorted(set(left_groups) & set(right_groups))
    if not shared_keys:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    left_obs_ids = left_frame["obs_id"].to_numpy(dtype=object)
    right_obs_ids = right_frame["obs_id"].to_numpy(dtype=object)
    left_plates = left_frame["plate"].to_numpy(dtype=object)
    right_plates = right_frame["plate"].to_numpy(dtype=object)
    left_wells = left_frame["well"].to_numpy(dtype=object)
    right_wells = right_frame["well"].to_numpy(dtype=object)
    left_log10_dose = left_frame["log10_dose"].to_numpy(dtype=np.float64)
    right_log10_dose = right_frame["log10_dose"].to_numpy(dtype=np.float64)
    left_dose_keys = left_frame["dose_key"].to_numpy(dtype=object)
    right_dose_keys = right_frame["dose_key"].to_numpy(dtype=object)

    context_matching_drugs: dict[tuple[str, str], set[str]] = defaultdict(set)
    rows: list[dict[str, object]] = []

    for pubchem_cid, cell_type, time_key in shared_keys:
        left_rows = left_groups[(pubchem_cid, cell_type, time_key)]
        right_rows = right_groups[(pubchem_cid, cell_type, time_key)]
        pairs = mutual_nearest_logdose_pairs(
            left_log10_doses=left_log10_dose[left_rows],
            right_log10_doses=right_log10_dose[right_rows],
            max_log10_dose_diff=max_log10_dose_diff,
        )
        if pairs.size == 0:
            continue

        for left_pos, right_pos in pairs:
            left_row = int(left_rows[left_pos])
            right_row = int(right_rows[right_pos])
            left_dose_key = str(left_dose_keys[left_row])
            right_dose_key = str(right_dose_keys[right_row])
            context_matching_drugs[(str(cell_type), str(time_key))].add(str(pubchem_cid))
            rows.append(
                {
                    "dataset_a": left_dataset,
                    "dataset_b": right_dataset,
                    "cell_type": str(cell_type),
                    "pubchem_cid": str(pubchem_cid),
                    "time_key": str(time_key),
                    "left_obs_id": str(left_obs_ids[left_row]),
                    "right_obs_id": str(right_obs_ids[right_row]),
                    "left_plate": str(left_plates[left_row]),
                    "right_plate": str(right_plates[right_row]),
                    "left_well": str(left_wells[left_row]),
                    "right_well": str(right_wells[right_row]),
                    "left_dose_key": left_dose_key,
                    "right_dose_key": right_dose_key,
                    "left_log10_dose": float(left_log10_dose[left_row]),
                    "right_log10_dose": float(right_log10_dose[right_row]),
                    "abs_delta_log10_dose": float(abs(left_log10_dose[left_row] - right_log10_dose[right_row])),
                    "matched_condition_key": "|".join(
                        [
                            str(pubchem_cid),
                            str(cell_type),
                            str(time_key),
                            left_dose_key,
                            right_dose_key,
                        ]
                    ),
                }
            )

    if not rows:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    frame = pd.DataFrame(rows)
    qualifying_context_drug_counts = {
        context_key: len(compounds)
        for context_key, compounds in context_matching_drugs.items()
        if len(compounds) >= int(min_context_shared_drugs)
    }
    if not qualifying_context_drug_counts:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    frame["_context_key"] = list(zip(frame["cell_type"], frame["time_key"]))
    frame["n_context_matching_drugs"] = frame["_context_key"].map(qualifying_context_drug_counts)
    frame = frame.loc[frame["n_context_matching_drugs"].notna()].copy()
    frame["n_context_matching_drugs"] = frame["n_context_matching_drugs"].astype(int)
    frame = frame.drop(columns="_context_key")
    return frame[MATCH_PAIR_COLUMNS].reset_index(drop=True)


In [55]:
@dataclass
class LineSource:
    dataset_name: str
    cell_type: str
    path: Path
    adata: ad.AnnData = field(init=False, repr=False)
    obs: pd.DataFrame = field(init=False, repr=False)
    unique_gene_keys: np.ndarray = field(init=False, repr=False)
    unique_gene_positions: np.ndarray = field(init=False, repr=False)
    gene_to_pos: dict[str, int] = field(init=False, repr=False)
    lookup_row_pos: dict[str, int] = field(init=False, repr=False)
    _vector_cache: dict[tuple[str, int], np.ndarray] = field(default_factory=dict, init=False, repr=False)
    _baseline_cache: dict[tuple[str, int], object] = field(default_factory=dict, init=False, repr=False)
    _baseline_peer_counts: dict[int, int] = field(default_factory=dict, init=False, repr=False)

    def __post_init__(self) -> None:
        self.adata = ad.read_h5ad(self.path, backed="r")
        obs = self.adata.obs.copy()
        row_positions = np.arange(self.adata.n_obs, dtype=np.int64)

        if "is_control" not in obs.columns:
            raise KeyError(f"{self.path} is missing obs['is_control']")

        obs["source_index"] = obs.index.astype(str)
        control_mask = coerce_control_mask(obs["is_control"]).to_numpy(dtype=bool)
        obs = obs.loc[~control_mask].copy()
        row_positions = row_positions[~control_mask]

        obs["source_row_pos"] = row_positions
        for column_name in [
            "id",
            "plate",
            "well",
            "cell_type",
            "perturbagen",
            "perturbagen_name",
            "perturbation_label",
            "pubchem_cid",
        ]:
            if column_name in obs.columns:
                obs[column_name] = obs[column_name].astype("string").fillna("").astype(str).str.strip()

        obs["pert_time_h"] = pd.to_numeric(obs["pert_time_h"], errors="coerce")
        obs["pert_dose_uM"] = pd.to_numeric(obs["pert_dose_uM"], errors="coerce")
        obs["time_key"] = obs["pert_time_h"].map(
            lambda value: format_numeric(float(value)) if pd.notna(value) else ""
        )
        obs["dose_key"] = obs["pert_dose_uM"].map(
            lambda value: format_numeric(float(value)) if pd.notna(value) and float(value) > 0 else ""
        )

        valid_mask = (
            (obs["pubchem_cid"] != "")
            & np.isfinite(obs["pert_time_h"].to_numpy(dtype=float))
            & np.isfinite(obs["pert_dose_uM"].to_numpy(dtype=float))
            & (obs["pert_dose_uM"].to_numpy(dtype=float) > 0.0)
        )
        obs = obs.loc[valid_mask].copy()
        obs = obs.set_index("source_row_pos", drop=False)
        self.obs = obs
        self.lookup_row_pos = self._build_lookup_row_pos()

        var = self.adata.var.copy()
        if "symbol" in var.columns:
            gene_key_series = pd.Series(
                var["symbol"].astype("string").fillna("").astype(str).str.strip().to_numpy(),
                index=np.arange(self.adata.n_vars, dtype=np.int64),
            )
        else:
            gene_key_series = pd.Series(
                pd.Index(self.adata.var_names.astype(str)).astype(str).str.strip().to_numpy(),
                index=np.arange(self.adata.n_vars, dtype=np.int64),
            )
        keep_mask = (gene_key_series != "") & ~gene_key_series.duplicated(keep="first")
        self.unique_gene_positions = gene_key_series.index[keep_mask].to_numpy(dtype=np.int64)
        self.unique_gene_keys = gene_key_series.loc[keep_mask].to_numpy(dtype=object)
        self.gene_to_pos = {
            str(gene_key): int(pos)
            for pos, gene_key in enumerate(self.unique_gene_keys.tolist())
        }

    def _build_lookup_row_pos(self) -> dict[str, int]:
        lookup_row_pos: dict[str, int] = {}

        def add_lookup_key(key: str, row_pos: int) -> None:
            if not key or key.lower() == "nan":
                return
            lookup_row_pos.setdefault(key, row_pos)

        for row_pos, row in self.obs.iterrows():
            row_pos = int(row_pos)
            for column_name in ["source_index", "id"]:
                if column_name in row.index:
                    add_lookup_key(str(row[column_name]).strip(), row_pos)

            cell_type = str(row.get("cell_type", "")).strip()
            pubchem_cid = str(row.get("pubchem_cid", "")).strip()
            dose_key = str(row.get("dose_key", "")).strip()
            time_key = str(row.get("time_key", "")).strip()

            if pubchem_cid and dose_key and time_key and cell_type:
                add_lookup_key(
                    f"{pubchem_cid}|{dose_key}|{time_key}|{cell_type}",
                    row_pos,
                )
        return lookup_row_pos

    def resolve_row_pos(
        self,
        obs_id: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ) -> int:
        obs_id = str(obs_id)
        candidate_keys = [obs_id]

        pubchem_cid = "" if pubchem_cid is None else str(pubchem_cid).strip()
        dose_key = "" if dose_key is None else str(dose_key).strip()
        time_key = "" if time_key is None else str(time_key).strip()

        if pubchem_cid and dose_key and time_key:
            candidate_keys.append(f"{pubchem_cid}|{dose_key}|{time_key}|{self.cell_type}")

        for candidate_key in candidate_keys:
            if candidate_key in self.lookup_row_pos:
                return int(self.lookup_row_pos[candidate_key])

        raise KeyError(
            f"Could not resolve obs_id={obs_id!r} in {self.path}. "
            f"Tried {candidate_keys!r}. Available lookup keys: {len(self.lookup_row_pos):,}"
        )

    def get_vector(
        self,
        obs_id: str,
        layer_name: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ) -> np.ndarray:
        row_pos = self.resolve_row_pos(
            obs_id,
            pubchem_cid=pubchem_cid,
            dose_key=dose_key,
            time_key=time_key,
            plate=plate,
            well=well,
        )
        cache_key = (layer_name, row_pos)
        if cache_key not in self._vector_cache:
            vector = np.asarray(self.adata.layers[layer_name][row_pos], dtype=np.float32).reshape(-1)
            self._vector_cache[cache_key] = vector[self.unique_gene_positions]
        return self._vector_cache[cache_key]

    def get_baseline_vector(
        self,
        obs_id: str,
        layer_name: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ):
        row_pos = self.resolve_row_pos(
            obs_id,
            pubchem_cid=pubchem_cid,
            dose_key=dose_key,
            time_key=time_key,
            plate=plate,
            well=well,
        )
        cache_key = (layer_name, row_pos)
        if cache_key not in self._baseline_cache:
            row = self.obs.loc[row_pos]
            peer_obs = self.obs.loc[
                (self.obs["dose_key"] == row["dose_key"])
                & (self.obs["time_key"] == row["time_key"])
                & (self.obs["pubchem_cid"] != row["pubchem_cid"])
            ]
            peer_rows = peer_obs["source_row_pos"].to_numpy(dtype=np.int64)
            self._baseline_peer_counts[row_pos] = int(len(peer_rows))
            if len(peer_rows) == 0:
                self._baseline_cache[cache_key] = None
            else:
                matrix = np.asarray(self.adata.layers[layer_name][peer_rows], dtype=np.float32)
                if matrix.ndim == 1:
                    matrix = matrix[np.newaxis, :]
                baseline = matrix[:, self.unique_gene_positions].mean(axis=0, dtype=np.float64)
                self._baseline_cache[cache_key] = np.asarray(baseline, dtype=np.float32)
        return self._baseline_cache[cache_key]

    def baseline_peer_count(
        self,
        obs_id: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ) -> int:
        row_pos = self.resolve_row_pos(
            obs_id,
            pubchem_cid=pubchem_cid,
            dose_key=dose_key,
            time_key=time_key,
            plate=plate,
            well=well,
        )
        if row_pos not in self._baseline_peer_counts:
            _ = self.get_baseline_vector(
                obs_id,
                "logFC",
                pubchem_cid=pubchem_cid,
                dose_key=dose_key,
                time_key=time_key,
                plate=plate,
                well=well,
            )
        return int(self._baseline_peer_counts.get(row_pos, 0))

    def close(self) -> None:
        self.adata.file.close()


def resolve_line_path(dataset_name: str, cell_type: str) -> Path:
    dataset_dir = SOURCE_DATASET_DIRS[dataset_name]
    candidates = [
        dataset_dir / f"{cell_type}_de.h5ad",
        dataset_dir / f"{cell_type}.h5ad",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find a line file for dataset={dataset_name}, cell_type={cell_type} in {dataset_dir}"
    )


LINE_SOURCE_CACHE: dict[tuple[str, str], LineSource] = {}
COMMON_GENE_CACHE: dict[tuple[str, str, str], tuple[np.ndarray, np.ndarray, np.ndarray]] = {}


def get_line_source(dataset_name: str, cell_type: str) -> LineSource:
    cache_key = (dataset_name, cell_type)
    if cache_key not in LINE_SOURCE_CACHE:
        LINE_SOURCE_CACHE[cache_key] = LineSource(
            dataset_name=dataset_name,
            cell_type=cell_type,
            path=resolve_line_path(dataset_name, cell_type),
        )
    return LINE_SOURCE_CACHE[cache_key]


def shared_gene_positions(
    left_source: LineSource,
    right_source: LineSource,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    cache_key = (left_source.dataset_name, right_source.dataset_name, left_source.cell_type)
    if cache_key not in COMMON_GENE_CACHE:
        shared_genes = [
            gene_key
            for gene_key in left_source.unique_gene_keys.tolist()
            if str(gene_key) in right_source.gene_to_pos
        ]
        left_positions = np.fromiter(
            (left_source.gene_to_pos[str(gene_key)] for gene_key in shared_genes),
            dtype=np.int64,
            count=len(shared_genes),
        )
        right_positions = np.fromiter(
            (right_source.gene_to_pos[str(gene_key)] for gene_key in shared_genes),
            dtype=np.int64,
            count=len(shared_genes),
        )
        COMMON_GENE_CACHE[cache_key] = (
            np.asarray(shared_genes, dtype=object),
            left_positions,
            right_positions,
        )
    return COMMON_GENE_CACHE[cache_key]

LINE_GLOBAL_SHARED_GENE_KEYS: dict[str, np.ndarray] = {}
GLOBAL_GENE_POSITION_CACHE: dict[tuple[str, str], np.ndarray] = {}


def set_global_shared_gene_keys(
    retained_lines: dict[str, list[str]],
    dataset_names: list[str],
) -> dict[str, np.ndarray]:
    global LINE_GLOBAL_SHARED_GENE_KEYS, GLOBAL_GENE_POSITION_CACHE

    cell_types = sorted(
        {
            cell_type
            for dataset_name in dataset_names
            for cell_type in retained_lines.get(dataset_name, [])
        }
    )
    line_gene_map: dict[str, np.ndarray] = {}
    for cell_type in cell_types:
        gene_sets: list[set[str]] = []
        for dataset_name in dataset_names:
            if cell_type not in retained_lines.get(dataset_name, []):
                continue
            source = get_line_source(dataset_name, cell_type)
            gene_sets.append({str(gene_key) for gene_key in source.unique_gene_keys.tolist()})
        if not gene_sets:
            line_gene_map[cell_type] = np.empty(0, dtype=object)
        else:
            line_gene_map[cell_type] = np.asarray(sorted(set.intersection(*gene_sets)), dtype=object)

    LINE_GLOBAL_SHARED_GENE_KEYS = line_gene_map
    GLOBAL_GENE_POSITION_CACHE = {}
    return LINE_GLOBAL_SHARED_GENE_KEYS


def global_gene_positions(source: LineSource) -> tuple[np.ndarray, np.ndarray]:
    cache_key = (source.dataset_name, source.cell_type)
    line_gene_keys = LINE_GLOBAL_SHARED_GENE_KEYS.get(source.cell_type, np.empty(0, dtype=object))
    if line_gene_keys.size == 0:
        return line_gene_keys, np.empty(0, dtype=np.int64)
    if cache_key not in GLOBAL_GENE_POSITION_CACHE:
        missing_gene_keys = [
            str(gene_key)
            for gene_key in line_gene_keys.tolist()
            if str(gene_key) not in source.gene_to_pos
        ]
        if missing_gene_keys:
            raise KeyError(
                f"Line-specific shared-gene set is inconsistent for {(source.dataset_name, source.cell_type)}; "
                f"missing {len(missing_gene_keys)} genes."
            )
        GLOBAL_GENE_POSITION_CACHE[cache_key] = np.fromiter(
            (source.gene_to_pos[str(gene_key)] for gene_key in line_gene_keys.tolist()),
            dtype=np.int64,
            count=int(line_gene_keys.size),
        )
    return line_gene_keys, GLOBAL_GENE_POSITION_CACHE[cache_key]



def filter_finite_pair(left_values: np.ndarray, right_values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mask = np.isfinite(left_values) & np.isfinite(right_values)
    return left_values[mask], right_values[mask]


def signed_spearman(left_values: np.ndarray, right_values: np.ndarray) -> float:
    left_values, right_values = filter_finite_pair(
        np.asarray(left_values, dtype=np.float64),
        np.asarray(right_values, dtype=np.float64),
    )
    if left_values.size < 2:
        return float("nan")

    left_ranks = rankdata(left_values, method="average")
    right_ranks = rankdata(right_values, method="average")
    if np.allclose(left_ranks, left_ranks[0]) or np.allclose(right_ranks, right_ranks[0]):
        return float("nan")
    return float(np.corrcoef(left_ranks, right_ranks)[0, 1])


def signed_overlap_at_k(left_values: np.ndarray, right_values: np.ndarray, *, k: int = TOP_K) -> float:
    left_values, right_values = filter_finite_pair(
        np.asarray(left_values, dtype=np.float64),
        np.asarray(right_values, dtype=np.float64),
    )
    n_genes = int(left_values.size)
    k_eff = min(int(k), n_genes // 2)
    if k_eff < 1:
        return float("nan")

    top_left = np.argpartition(left_values, -k_eff)[-k_eff:]
    top_right = np.argpartition(right_values, -k_eff)[-k_eff:]
    bottom_left = np.argpartition(left_values, k_eff - 1)[:k_eff]
    bottom_right = np.argpartition(right_values, k_eff - 1)[:k_eff]

    top_overlap = np.intersect1d(top_left, top_right, assume_unique=False).size / k_eff
    bottom_overlap = np.intersect1d(bottom_left, bottom_right, assume_unique=False).size / k_eff
    return float(0.5 * (top_overlap + bottom_overlap))


def score_signature_pair(
    left_logfc: np.ndarray,
    right_logfc: np.ndarray,
    left_t: np.ndarray,
    right_t: np.ndarray,
) -> dict[str, float]:
    return {
        "spearman_logfc": signed_spearman(left_logfc, right_logfc),
        "spearman_t": signed_spearman(left_t, right_t),
        f"signed_overlap_t_top{TOP_K}": signed_overlap_at_k(left_t, right_t, k=TOP_K),
    }


def mean_available(values: list[float]) -> float:
    finite_values = [value for value in values if pd.notna(value)]
    if not finite_values:
        return float("nan")
    return float(np.mean(finite_values))

def empty_score_dict() -> dict[str, float]:
    return {
        "spearman_logfc": float("nan"),
        "spearman_t": float("nan"),
        f"signed_overlap_t_top{TOP_K}": float("nan"),
    }



def compute_metric_record(match_row: pd.Series) -> dict[str, object]:
    left_source = get_line_source(str(match_row["dataset_a"]), str(match_row["cell_type"]))
    right_source = get_line_source(str(match_row["dataset_b"]), str(match_row["cell_type"]))
    shared_genes, left_gene_pos, right_gene_pos = shared_gene_positions(left_source, right_source)
    if shared_genes.size < 2:
        raise ValueError(
            f"Fewer than two shared genes for {match_row['dataset_a']} vs {match_row['dataset_b']} / {match_row['cell_type']}"
        )

    global_shared_genes, left_global_gene_pos = global_gene_positions(left_source)
    _, right_global_gene_pos = global_gene_positions(right_source)

    left_obs_id = str(match_row["left_obs_id"])
    right_obs_id = str(match_row["right_obs_id"])
    pubchem_cid = str(match_row["pubchem_cid"])
    time_key = str(match_row["time_key"])

    left_lookup = {
        "pubchem_cid": pubchem_cid,
        "dose_key": str(match_row["left_dose_key"]),
        "time_key": time_key,
    }
    right_lookup = {
        "pubchem_cid": pubchem_cid,
        "dose_key": str(match_row["right_dose_key"]),
        "time_key": time_key,
    }

    left_logfc_full = left_source.get_vector(left_obs_id, "logFC", **left_lookup)
    right_logfc_full = right_source.get_vector(right_obs_id, "logFC", **right_lookup)
    left_t_full = left_source.get_vector(left_obs_id, "t", **left_lookup)
    right_t_full = right_source.get_vector(right_obs_id, "t", **right_lookup)

    observed_scores = score_signature_pair(
        left_logfc=left_logfc_full[left_gene_pos],
        right_logfc=right_logfc_full[right_gene_pos],
        left_t=left_t_full[left_gene_pos],
        right_t=right_t_full[right_gene_pos],
    )

    observed_scores_global = empty_score_dict()
    if global_shared_genes.size >= 2:
        observed_scores_global = score_signature_pair(
            left_logfc=left_logfc_full[left_global_gene_pos],
            right_logfc=right_logfc_full[right_global_gene_pos],
            left_t=left_t_full[left_global_gene_pos],
            right_t=right_t_full[right_global_gene_pos],
        )

    left_mean_abs_t = mean_available(np.abs(left_t_full[left_gene_pos]).tolist())
    right_mean_abs_t = mean_available(np.abs(right_t_full[right_gene_pos]).tolist())
    pair_mean_abs_t = mean_available([left_mean_abs_t, right_mean_abs_t])

    left_mean_abs_t_global = float("nan")
    right_mean_abs_t_global = float("nan")
    pair_mean_abs_t_global = float("nan")
    if global_shared_genes.size >= 2:
        left_mean_abs_t_global = mean_available(np.abs(left_t_full[left_global_gene_pos]).tolist())
        right_mean_abs_t_global = mean_available(np.abs(right_t_full[right_global_gene_pos]).tolist())
        pair_mean_abs_t_global = mean_available([left_mean_abs_t_global, right_mean_abs_t_global])

    left_baseline_logfc_full = left_source.get_baseline_vector(left_obs_id, "logFC", **left_lookup)
    left_baseline_t_full = left_source.get_baseline_vector(left_obs_id, "t", **left_lookup)
    right_baseline_logfc_full = right_source.get_baseline_vector(right_obs_id, "logFC", **right_lookup)
    right_baseline_t_full = right_source.get_baseline_vector(right_obs_id, "t", **right_lookup)

    left_baseline_scores = empty_score_dict()
    left_baseline_scores_global = empty_score_dict()
    if left_baseline_logfc_full is not None and left_baseline_t_full is not None:
        left_baseline_scores = score_signature_pair(
            left_logfc=left_logfc_full[left_gene_pos],
            right_logfc=left_baseline_logfc_full[left_gene_pos],
            left_t=left_t_full[left_gene_pos],
            right_t=left_baseline_t_full[left_gene_pos],
        )
        if global_shared_genes.size >= 2:
            left_baseline_scores_global = score_signature_pair(
                left_logfc=left_logfc_full[left_global_gene_pos],
                right_logfc=left_baseline_logfc_full[left_global_gene_pos],
                left_t=left_t_full[left_global_gene_pos],
                right_t=left_baseline_t_full[left_global_gene_pos],
            )

    right_baseline_scores = empty_score_dict()
    right_baseline_scores_global = empty_score_dict()
    if right_baseline_logfc_full is not None and right_baseline_t_full is not None:
        right_baseline_scores = score_signature_pair(
            left_logfc=right_logfc_full[right_gene_pos],
            right_logfc=right_baseline_logfc_full[right_gene_pos],
            left_t=right_t_full[right_gene_pos],
            right_t=right_baseline_t_full[right_gene_pos],
        )
        if global_shared_genes.size >= 2:
            right_baseline_scores_global = score_signature_pair(
                left_logfc=right_logfc_full[right_global_gene_pos],
                right_logfc=right_baseline_logfc_full[right_global_gene_pos],
                left_t=right_t_full[right_global_gene_pos],
                right_t=right_baseline_t_full[right_global_gene_pos],
            )

    overlap_column = f"signed_overlap_t_top{TOP_K}"
    return {
        **match_row.to_dict(),
        "n_common_genes": int(shared_genes.size),
        "n_global_common_genes": int(global_shared_genes.size),
        "left_mean_abs_t": left_mean_abs_t,
        "right_mean_abs_t": right_mean_abs_t,
        "pair_mean_abs_t": pair_mean_abs_t,
        "left_mean_abs_t_global": left_mean_abs_t_global,
        "right_mean_abs_t_global": right_mean_abs_t_global,
        "pair_mean_abs_t_global": pair_mean_abs_t_global,
        "observed_spearman_logfc": observed_scores["spearman_logfc"],
        "observed_spearman_logfc_global": observed_scores_global["spearman_logfc"],
        "observed_spearman_t": observed_scores["spearman_t"],
        "observed_spearman_t_global": observed_scores_global["spearman_t"],
        f"observed_{overlap_column}": observed_scores[overlap_column],
        f"observed_{overlap_column}_global": observed_scores_global[overlap_column],
        "left_baseline_peer_count": left_source.baseline_peer_count(left_obs_id, **left_lookup),
        "left_baseline_spearman_logfc": left_baseline_scores["spearman_logfc"],
        "left_baseline_spearman_logfc_global": left_baseline_scores_global["spearman_logfc"],
        "left_baseline_spearman_t": left_baseline_scores["spearman_t"],
        "left_baseline_spearman_t_global": left_baseline_scores_global["spearman_t"],
        f"left_baseline_{overlap_column}": left_baseline_scores[overlap_column],
        f"left_baseline_{overlap_column}_global": left_baseline_scores_global[overlap_column],
        "right_baseline_peer_count": right_source.baseline_peer_count(right_obs_id, **right_lookup),
        "right_baseline_spearman_logfc": right_baseline_scores["spearman_logfc"],
        "right_baseline_spearman_logfc_global": right_baseline_scores_global["spearman_logfc"],
        "right_baseline_spearman_t": right_baseline_scores["spearman_t"],
        "right_baseline_spearman_t_global": right_baseline_scores_global["spearman_t"],
        f"right_baseline_{overlap_column}": right_baseline_scores[overlap_column],
        f"right_baseline_{overlap_column}_global": right_baseline_scores_global[overlap_column],
        "baseline_pair_mean_spearman_logfc": mean_available(
            [left_baseline_scores["spearman_logfc"], right_baseline_scores["spearman_logfc"]]
        ),
        "baseline_pair_mean_spearman_logfc_global": mean_available(
            [left_baseline_scores_global["spearman_logfc"], right_baseline_scores_global["spearman_logfc"]]
        ),
        "baseline_pair_mean_spearman_t": mean_available(
            [left_baseline_scores["spearman_t"], right_baseline_scores["spearman_t"]]
        ),
        "baseline_pair_mean_spearman_t_global": mean_available(
            [left_baseline_scores_global["spearman_t"], right_baseline_scores_global["spearman_t"]]
        ),
        f"baseline_pair_mean_{overlap_column}": mean_available(
            [left_baseline_scores[overlap_column], right_baseline_scores[overlap_column]]
        ),
        f"baseline_pair_mean_{overlap_column}_global": mean_available(
            [left_baseline_scores_global[overlap_column], right_baseline_scores_global[overlap_column]]
        ),
    }

def close_all_line_sources() -> None:
    for line_source in LINE_SOURCE_CACHE.values():
        line_source.close()


In [56]:
dataset_indices = {dataset_name: build_dataset_index(dataset_name) for dataset_name in DATASET_ORDER}
active_datasets = active_dataset_names(dataset_indices)
if not active_datasets:
    raise ValueError("No overlap-filtered non-control samples were found for the configured datasets.")

print("Datasets in scope:", ", ".join(pretty_label(dataset_name) for dataset_name in active_datasets))

retained_lines = {
    dataset_name: sorted(dataset_indices[dataset_name]["frame"]["cell_type"].unique().tolist())
    for dataset_name in active_datasets
}
retained_lines_display = pd.DataFrame(
    {
        "dataset": [pretty_label(dataset_name) for dataset_name in retained_lines],
        "cell_types": [", ".join(lines) for lines in retained_lines.values()],
    }
)
display(retained_lines_display)

line_global_gene_keys = set_global_shared_gene_keys(retained_lines, active_datasets)
line_global_gene_counts = {
    cell_type: int(gene_keys.size)
    for cell_type, gene_keys in line_global_gene_keys.items()
}
print(f"Line-specific shared-gene sets computed for {len(line_global_gene_counts):,} retained lines.")
if not line_global_gene_counts or max(line_global_gene_counts.values()) < 2:
    print(
        "Line-specific shared-gene evaluation will be unavailable because no retained line has at least two genes shared across the datasets that retain it."
    )
else:
    print(
        f"Line-specific shared-gene count range across retained lines: {min(line_global_gene_counts.values()):,} to {max(line_global_gene_counts.values()):,}"
    )

pair_match_frames: list[pd.DataFrame] = []
for dataset_a, dataset_b in itertools.combinations(active_datasets, 2):
    frame = pair_match_frame(
        left_dataset=dataset_a,
        right_dataset=dataset_b,
        left_index=dataset_indices[dataset_a],
        right_index=dataset_indices[dataset_b],
    )
    if not frame.empty:
        pair_match_frames.append(frame)

if not pair_match_frames:
    raise ValueError("No matched grouped-replicate sample pairs were found.")

matched_pairs = pd.concat(pair_match_frames, ignore_index=True)
matched_pairs_path = OUTPUT_DIR / "matched_sample_pairs.tsv"
matched_pairs.to_csv(matched_pairs_path, sep="\t", index=False)
print(f"Saved matched sample pairs to {matched_pairs_path}")

pair_match_summary = (
    matched_pairs.groupby(["dataset_a", "dataset_b"], as_index=False)
    .agg(
        n_matched_sample_pairs=("left_obs_id", "size"),
        n_matching_drugs=("pubchem_cid", "nunique"),
        n_matching_lines=("cell_type", "nunique"),
        n_matching_conditions=("matched_condition_key", "nunique"),
    )
)
pair_match_summary_display = pair_match_summary.copy()
pair_match_summary_display["dataset_a"] = pair_match_summary_display["dataset_a"].map(pretty_label)
pair_match_summary_display["dataset_b"] = pair_match_summary_display["dataset_b"].map(pretty_label)
display(pair_match_summary_display)

line_match_summary = (
    matched_pairs.groupby(["dataset_a", "dataset_b", "cell_type"], as_index=False)
    .agg(
        n_matched_sample_pairs=("left_obs_id", "size"),
        n_matching_drugs=("pubchem_cid", "nunique"),
        n_matching_conditions=("matched_condition_key", "nunique"),
    )
    .sort_values(["dataset_a", "dataset_b", "cell_type"]) 
    .reset_index(drop=True)
)
line_match_summary_display = line_match_summary.copy()
line_match_summary_display["dataset_a"] = line_match_summary_display["dataset_a"].map(pretty_label)
line_match_summary_display["dataset_b"] = line_match_summary_display["dataset_b"].map(pretty_label)
display(line_match_summary_display)


Datasets in scope: L1000 Phase I, L1000 Phase II, sci-Plex, Tahoe-100M


,dataset,cell_types
0,L1000 Phase I,"CVCL_0023, CVCL_0031, CVCL_0320, CVCL_0332"
1,L1000 Phase II,"CVCL_0023, CVCL_0031, CVCL_0320"
2,sci-Plex,"CVCL_0023, CVCL_0031"
3,Tahoe-100M,"CVCL_0023, CVCL_0320, CVCL_0332"


/ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.o

Line-specific shared-gene sets computed for 4 retained lines.
Line-specific shared-gene count range across retained lines: 599 to 941
Saved matched sample pairs to /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_group_rep_retrieval_metrics/matched_sample_pairs.tsv


,dataset_a,dataset_b,n_matched_sample_pairs,n_matching_drugs,n_matching_lines,n_matching_conditions
0,L1000 Phase I,L1000 Phase II,10855,190,3,375
1,L1000 Phase I,sci-Plex,2695,90,2,291
2,L1000 Phase I,Tahoe-100M,610,86,3,152
3,L1000 Phase II,sci-Plex,3192,83,2,381
4,L1000 Phase II,Tahoe-100M,1473,110,2,402
5,sci-Plex,Tahoe-100M,170,23,1,69


,dataset_a,dataset_b,cell_type,n_matched_sample_pairs,n_matching_drugs,n_matching_conditions
0,L1000 Phase I,L1000 Phase II,CVCL_0023,8049,152,237
1,L1000 Phase I,L1000 Phase II,CVCL_0031,2656,57,124
2,L1000 Phase I,L1000 Phase II,CVCL_0320,150,12,14
3,L1000 Phase I,sci-Plex,CVCL_0023,904,75,113
4,L1000 Phase I,sci-Plex,CVCL_0031,1791,90,178
5,L1000 Phase I,Tahoe-100M,CVCL_0023,402,83,95
6,L1000 Phase I,Tahoe-100M,CVCL_0320,96,20,24
7,L1000 Phase I,Tahoe-100M,CVCL_0332,112,11,33
8,L1000 Phase II,sci-Plex,CVCL_0023,1400,44,132
9,L1000 Phase II,sci-Plex,CVCL_0031,1792,83,249


**Retrieval Definitions**

For a fixed dataset pair and matched `cell_type + time_key` stratum, every retained condition in dataset A is queried against all retained conditions in dataset B within that same stratum.

The primary task is compound retrieval across doses. For query `i`, the positive set is all target-side conditions with the same `pubchem_cid`.

Scores:
- normalized best-positive rank: `1` is best, `0.5` is random on average, `0` is worst
- Recall@1
- AUROC with same-compound conditions as positives and other compounds as negatives

Candidates are scored by negative Euclidean distance on the chosen representation vector, so larger values mean smaller `L2` distance.


In [57]:
RETRIEVAL_VARIANTS = {
    "compound_across_doses": "Compound retrieval across doses",
    "dose_aware_compound": "Dose-aware compound retrieval",
    "strict_matched_condition": "Strict matched-condition retrieval",
}
REPRESENTATION_LABELS = {
    "signed_significance": "Signed significance",
    "moderated_t": "Moderated t",
    "logFC": "logFC",
}
MIN_TARGET_CANDIDATES = 5
MIN_UNIQUE_COMPOUNDS_PER_SIDE = 2
ADJ_PVALUE_LAYER_PREFERENCES = (
    "adj.P.Value.within_one_contrast",
    "adj.P.Value.across_all_contrasts",
)
ADJ_PVALUE_LAYER_CACHE: dict[tuple[str, str], str] = {}


def get_adjusted_pvalue_layer(source: LineSource) -> str:
    cache_key = (source.dataset_name, source.cell_type)
    if cache_key not in ADJ_PVALUE_LAYER_CACHE:
        available_layers = set(source.adata.layers.keys())
        for candidate in ADJ_PVALUE_LAYER_PREFERENCES:
            if candidate in available_layers:
                ADJ_PVALUE_LAYER_CACHE[cache_key] = candidate
                break
        else:
            raise KeyError(
                f"None of {ADJ_PVALUE_LAYER_PREFERENCES!r} are present in {source.path}; available layers: {sorted(available_layers)!r}"
            )
    return ADJ_PVALUE_LAYER_CACHE[cache_key]


def signed_significance(logfc: np.ndarray, adj_p: np.ndarray) -> np.ndarray:
    adj_p = np.asarray(adj_p, dtype=np.float64)
    logfc = np.asarray(logfc, dtype=np.float64)
    return -np.log10(np.clip(adj_p, 1e-300, None)) * np.sign(logfc)


def negative_l2_similarity_matrix(query_matrix: np.ndarray, candidate_matrix: np.ndarray) -> np.ndarray:
    query_matrix = np.asarray(query_matrix, dtype=np.float64)
    candidate_matrix = np.asarray(candidate_matrix, dtype=np.float64)
    diff = query_matrix[:, None, :] - candidate_matrix[None, :, :]
    with np.errstate(invalid="ignore"):
        distances = np.sqrt(np.sum(diff * diff, axis=2))
    return -distances


def normalized_best_positive_rank(scores: np.ndarray, positive_mask: np.ndarray) -> tuple[float, float]:
    positive_mask = np.asarray(positive_mask, dtype=bool)
    scores = np.asarray(scores, dtype=np.float64)
    if scores.size < 2 or int(positive_mask.sum()) == 0:
        return float("nan"), float("nan")
    ranks = rankdata(-scores, method="min")
    best_rank = float(np.min(ranks[positive_mask]))
    normalized_rank = 1.0 - ((best_rank - 1.0) / (len(scores) - 1.0))
    return float(best_rank), float(normalized_rank)


def recall_at_1(scores: np.ndarray, positive_mask: np.ndarray) -> float:
    positive_mask = np.asarray(positive_mask, dtype=bool)
    scores = np.asarray(scores, dtype=np.float64)
    if scores.size == 0 or int(positive_mask.sum()) == 0:
        return float("nan")
    ranks = rankdata(-scores, method="min")
    return float(np.min(ranks[positive_mask]) == 1)


def auroc_from_scores(scores: np.ndarray, positive_mask: np.ndarray) -> float:
    positive_mask = np.asarray(positive_mask, dtype=bool)
    scores = np.asarray(scores, dtype=np.float64)
    positive_scores = scores[positive_mask]
    negative_scores = scores[~positive_mask]
    if positive_scores.size == 0 or negative_scores.size == 0:
        return float("nan")
    wins = 0.0
    total = float(positive_scores.size * negative_scores.size)
    for positive_score in positive_scores:
        wins += float(np.sum(positive_score > negative_scores))
        wins += 0.5 * float(np.sum(np.isclose(positive_score, negative_scores, rtol=0.0, atol=1e-12)))
    return float(wins / total)


def build_strict_positive_maps(matched_pairs: pd.DataFrame) -> dict[tuple[str, str, str], dict[str, set[str]]]:
    maps: dict[tuple[str, str, str], dict[str, set[str]]] = {}
    for (dataset_a, dataset_b), group in matched_pairs.groupby(["dataset_a", "dataset_b"], sort=False):
        forward: dict[str, set[str]] = defaultdict(set)
        reverse: dict[str, set[str]] = defaultdict(set)
        for _, row in group.iterrows():
            forward[str(row["left_obs_id"])].add(str(row["right_obs_id"]))
            reverse[str(row["right_obs_id"])].add(str(row["left_obs_id"]))
        maps[(str(dataset_a), str(dataset_b), "A_to_B")] = forward
        maps[(str(dataset_a), str(dataset_b), "B_to_A")] = reverse
    return maps


def pool_frame_for_context(dataset_frame: pd.DataFrame, cell_type: str, time_key: str) -> pd.DataFrame:
    return dataset_frame.loc[
        (dataset_frame["cell_type"].astype(str) == str(cell_type))
        & (dataset_frame["time_key"].astype(str) == str(time_key))
    ].copy().reset_index(drop=True)


def build_pool_matrices(
    pool_frame: pd.DataFrame,
    source: LineSource,
    gene_positions: np.ndarray,
    *,
    adj_layer_name: str,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray, list[dict[str, object]]]:
    resolved_rows: list[dict[str, object]] = []
    unresolved_records: list[dict[str, object]] = []
    logfc_rows: list[np.ndarray] = []
    t_rows: list[np.ndarray] = []
    adj_p_rows: list[np.ndarray] = []
    for _, row in pool_frame.iterrows():
        lookup = {
            "pubchem_cid": str(row["pubchem_cid"]),
            "dose_key": str(row["dose_key"]),
            "time_key": str(row["time_key"]),
        }
        obs_id = str(row["obs_id"])
        try:
            logfc_rows.append(source.get_vector(obs_id, "logFC", **lookup)[gene_positions])
            t_rows.append(source.get_vector(obs_id, "t", **lookup)[gene_positions])
            adj_p_rows.append(source.get_vector(obs_id, adj_layer_name, **lookup)[gene_positions])
            resolved_rows.append(row.to_dict())
        except KeyError as exc:
            unresolved_records.append({
                **row.to_dict(),
                "source_dataset": source.dataset_name,
                "source_cell_type": source.cell_type,
                "source_path": str(source.path),
                "adj_pvalue_layer": adj_layer_name,
                "error": str(exc),
            })
    if not logfc_rows:
        return (
            pool_frame.iloc[0:0].copy(),
            np.empty((0, 0), dtype=np.float64),
            np.empty((0, 0), dtype=np.float64),
            np.empty((0, 0), dtype=np.float64),
            unresolved_records,
        )
    return (
        pd.DataFrame(resolved_rows).reset_index(drop=True),
        np.vstack(logfc_rows).astype(np.float64),
        np.vstack(t_rows).astype(np.float64),
        np.vstack(adj_p_rows).astype(np.float64),
        unresolved_records,
    )


def primary_positive_mask(query_compound: str, candidate_compounds: np.ndarray) -> np.ndarray:
    return np.asarray(candidate_compounds, dtype=object) == str(query_compound)


def dose_aware_positive_mask(
    query_compound: str,
    query_log10_dose: float,
    candidate_compounds: np.ndarray,
    candidate_log10_doses: np.ndarray,
) -> np.ndarray:
    same_compound = np.asarray(candidate_compounds, dtype=object) == str(query_compound)
    if int(same_compound.sum()) == 0:
        return np.zeros(len(candidate_compounds), dtype=bool)
    dose_diff = np.abs(np.asarray(candidate_log10_doses, dtype=np.float64) - float(query_log10_dose))
    within = same_compound & (dose_diff <= MAX_LOG10_DOSE_DIFF + 1e-12)
    if int(within.sum()) > 0:
        return within
    nearest_diff = float(np.min(dose_diff[same_compound]))
    return same_compound & np.isclose(dose_diff, nearest_diff, rtol=0.0, atol=1e-12)


def strict_positive_mask(
    query_obs_id: str,
    candidate_obs_ids: np.ndarray,
    strict_map: dict[str, set[str]],
) -> np.ndarray:
    positives = strict_map.get(str(query_obs_id), set())
    if not positives:
        return np.zeros(len(candidate_obs_ids), dtype=bool)
    return np.asarray([str(obs_id) in positives for obs_id in candidate_obs_ids], dtype=bool)


def representation_matrices_from_raw(
    left_logfc: np.ndarray,
    right_logfc: np.ndarray,
    left_t: np.ndarray,
    right_t: np.ndarray,
    left_adj_p: np.ndarray,
    right_adj_p: np.ndarray,
) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    return {
        "signed_significance": (
            signed_significance(left_logfc, left_adj_p),
            signed_significance(right_logfc, right_adj_p),
        ),
        "moderated_t": (left_t, right_t),
        "logFC": (left_logfc, right_logfc),
    }


In [58]:
strict_positive_maps = build_strict_positive_maps(matched_pairs)
retrieval_records: list[dict[str, object]] = []
skipped_context_records: list[dict[str, object]] = []
unresolved_pool_records: list[dict[str, object]] = []

pair_contexts = (
    matched_pairs[["dataset_a", "dataset_b", "cell_type", "time_key"]]
    .drop_duplicates()
    .sort_values(["dataset_a", "dataset_b", "cell_type", "time_key"])
    .reset_index(drop=True)
)

for _, context_row in pair_contexts.iterrows():
    dataset_a = str(context_row["dataset_a"])
    dataset_b = str(context_row["dataset_b"])
    cell_type = str(context_row["cell_type"])
    time_key = str(context_row["time_key"])

    left_pool = pool_frame_for_context(dataset_indices[dataset_a]["frame"], cell_type, time_key)
    right_pool = pool_frame_for_context(dataset_indices[dataset_b]["frame"], cell_type, time_key)

    left_unique_compounds = int(left_pool["pubchem_cid"].astype(str).nunique())
    right_unique_compounds = int(right_pool["pubchem_cid"].astype(str).nunique())
    if (
        left_unique_compounds < MIN_UNIQUE_COMPOUNDS_PER_SIDE
        or right_unique_compounds < MIN_UNIQUE_COMPOUNDS_PER_SIDE
        or len(left_pool) < MIN_TARGET_CANDIDATES
        or len(right_pool) < MIN_TARGET_CANDIDATES
    ):
        skipped_context_records.append(
            {
                **context_row.to_dict(),
                "reason": "insufficient_candidates_or_unique_compounds",
                "n_left_conditions": int(len(left_pool)),
                "n_right_conditions": int(len(right_pool)),
                "n_left_unique_compounds": left_unique_compounds,
                "n_right_unique_compounds": right_unique_compounds,
            }
        )
        continue

    left_source = get_line_source(dataset_a, cell_type)
    right_source = get_line_source(dataset_b, cell_type)
    shared_genes, left_gene_pos, right_gene_pos = shared_gene_positions(left_source, right_source)
    if shared_genes.size < 2:
        skipped_context_records.append(
            {
                **context_row.to_dict(),
                "reason": "too_few_shared_genes",
                "n_shared_genes": int(shared_genes.size),
            }
        )
        continue

    left_adj_layer = get_adjusted_pvalue_layer(left_source)
    right_adj_layer = get_adjusted_pvalue_layer(right_source)
    left_pool, left_logfc, left_t, left_adj_p, left_unresolved = build_pool_matrices(
        left_pool,
        left_source,
        left_gene_pos,
        adj_layer_name=left_adj_layer,
    )
    right_pool, right_logfc, right_t, right_adj_p, right_unresolved = build_pool_matrices(
        right_pool,
        right_source,
        right_gene_pos,
        adj_layer_name=right_adj_layer,
    )
    unresolved_pool_records.extend(
        [
            {
                **record,
                "dataset_a": dataset_a,
                "dataset_b": dataset_b,
                "cell_type": cell_type,
                "time_key": time_key,
                "pool_side": "left",
            }
            for record in left_unresolved
        ]
    )
    unresolved_pool_records.extend(
        [
            {
                **record,
                "dataset_a": dataset_a,
                "dataset_b": dataset_b,
                "cell_type": cell_type,
                "time_key": time_key,
                "pool_side": "right",
            }
            for record in right_unresolved
        ]
    )

    left_unique_compounds = int(left_pool["pubchem_cid"].astype(str).nunique())
    right_unique_compounds = int(right_pool["pubchem_cid"].astype(str).nunique())
    if (
        left_unique_compounds < MIN_UNIQUE_COMPOUNDS_PER_SIDE
        or right_unique_compounds < MIN_UNIQUE_COMPOUNDS_PER_SIDE
        or len(left_pool) < MIN_TARGET_CANDIDATES
        or len(right_pool) < MIN_TARGET_CANDIDATES
    ):
        skipped_context_records.append(
            {
                **context_row.to_dict(),
                "reason": "insufficient_resolved_candidates_or_unique_compounds",
                "n_left_conditions": int(len(left_pool)),
                "n_right_conditions": int(len(right_pool)),
                "n_left_unique_compounds": left_unique_compounds,
                "n_right_unique_compounds": right_unique_compounds,
                "n_left_unresolved": int(len(left_unresolved)),
                "n_right_unresolved": int(len(right_unresolved)),
            }
        )
        continue

    finite_gene_mask = (
        np.isfinite(left_logfc).all(axis=0)
        & np.isfinite(right_logfc).all(axis=0)
        & np.isfinite(left_t).all(axis=0)
        & np.isfinite(right_t).all(axis=0)
        & np.isfinite(left_adj_p).all(axis=0)
        & np.isfinite(right_adj_p).all(axis=0)
    )
    shared_genes = shared_genes[finite_gene_mask]
    left_logfc = left_logfc[:, finite_gene_mask]
    right_logfc = right_logfc[:, finite_gene_mask]
    left_t = left_t[:, finite_gene_mask]
    right_t = right_t[:, finite_gene_mask]
    left_adj_p = left_adj_p[:, finite_gene_mask]
    right_adj_p = right_adj_p[:, finite_gene_mask]

    if shared_genes.size < 2:
        skipped_context_records.append(
            {
                **context_row.to_dict(),
                "reason": "too_few_finite_shared_genes",
                "n_shared_genes": int(shared_genes.size),
            }
        )
        continue

    representations = representation_matrices_from_raw(
        left_logfc,
        right_logfc,
        left_t,
        right_t,
        left_adj_p,
        right_adj_p,
    )

    directional_specs = [
        {
            "direction": "A_to_B",
            "query_dataset": dataset_a,
            "target_dataset": dataset_b,
            "query_pool": left_pool,
            "target_pool": right_pool,
            "query_side": 0,
            "target_side": 1,
            "strict_map": strict_positive_maps.get((dataset_a, dataset_b, "A_to_B"), {}),
        },
        {
            "direction": "B_to_A",
            "query_dataset": dataset_b,
            "target_dataset": dataset_a,
            "query_pool": right_pool,
            "target_pool": left_pool,
            "query_side": 1,
            "target_side": 0,
            "strict_map": strict_positive_maps.get((dataset_a, dataset_b, "B_to_A"), {}),
        },
    ]

    for direction_spec in directional_specs:
        query_pool = direction_spec["query_pool"].reset_index(drop=True)
        target_pool = direction_spec["target_pool"].reset_index(drop=True)
        if len(target_pool) < MIN_TARGET_CANDIDATES:
            continue

        query_compounds = query_pool["pubchem_cid"].astype(str).to_numpy(dtype=object)
        target_compounds = target_pool["pubchem_cid"].astype(str).to_numpy(dtype=object)
        query_obs_ids = query_pool["obs_id"].astype(str).to_numpy(dtype=object)
        target_obs_ids = target_pool["obs_id"].astype(str).to_numpy(dtype=object)
        query_log10_doses = query_pool["log10_dose"].to_numpy(dtype=np.float64)
        target_log10_doses = target_pool["log10_dose"].to_numpy(dtype=np.float64)

        for representation_name, (left_matrix, right_matrix) in representations.items():
            if direction_spec["query_side"] == 0:
                query_matrix = left_matrix
                target_matrix = right_matrix
            else:
                query_matrix = right_matrix
                target_matrix = left_matrix

            similarity = negative_l2_similarity_matrix(query_matrix, target_matrix)
            for query_idx in range(len(query_pool)):
                query_compound = str(query_compounds[query_idx])
                query_obs_id = str(query_obs_ids[query_idx])
                query_log10_dose = float(query_log10_doses[query_idx])
                query_scores = similarity[query_idx]

                positive_masks = {
                    "compound_across_doses": primary_positive_mask(query_compound, target_compounds),
                    "dose_aware_compound": dose_aware_positive_mask(
                        query_compound,
                        query_log10_dose,
                        target_compounds,
                        target_log10_doses,
                    ),
                    "strict_matched_condition": strict_positive_mask(
                        query_obs_id,
                        target_obs_ids,
                        direction_spec["strict_map"],
                    ),
                }

                for retrieval_variant, positive_mask in positive_masks.items():
                    n_positives = int(np.sum(positive_mask))
                    n_negatives = int(len(positive_mask) - n_positives)
                    if n_positives == 0 or n_negatives == 0:
                        continue
                    best_rank, normalized_rank = normalized_best_positive_rank(query_scores, positive_mask)
                    retrieval_records.append(
                        {
                            "dataset_a": dataset_a,
                            "dataset_b": dataset_b,
                            "direction": direction_spec["direction"],
                            "query_dataset": direction_spec["query_dataset"],
                            "target_dataset": direction_spec["target_dataset"],
                            "cell_type": cell_type,
                            "time_key": time_key,
                            "representation": representation_name,
                            "retrieval_variant": retrieval_variant,
                            "query_obs_id": query_obs_id,
                            "query_pubchem_cid": query_compound,
                            "query_dose_key": str(query_pool.loc[query_idx, "dose_key"]),
                            "query_log10_dose": query_log10_dose,
                            "target_pool_size": int(len(target_pool)),
                            "n_query_side_conditions": int(len(query_pool)),
                            "n_target_side_conditions": int(len(target_pool)),
                            "n_query_side_unique_compounds": int(query_pool["pubchem_cid"].astype(str).nunique()),
                            "n_target_side_unique_compounds": int(target_pool["pubchem_cid"].astype(str).nunique()),
                            "n_shared_genes": int(shared_genes.size),
                            "n_positive_candidates": n_positives,
                            "best_positive_rank": best_rank,
                            "normalized_best_positive_rank": normalized_rank,
                            "recall_at_1": recall_at_1(query_scores, positive_mask),
                            "auroc": auroc_from_scores(query_scores, positive_mask),
                            "random_recall_at_1": float(n_positives / len(target_pool)),
                            "random_normalized_best_positive_rank": 0.5,
                            "random_auroc": 0.5,
                        }
                    )

query_retrieval_metrics = pd.DataFrame(retrieval_records)
if query_retrieval_metrics.empty:
    raise ValueError("No eligible retrieval queries were scored.")

query_retrieval_metrics_path = OUTPUT_DIR / "query_retrieval_metrics.tsv"
query_retrieval_metrics.to_csv(query_retrieval_metrics_path, sep="	", index=False)
print(f"Saved query-level retrieval metrics to {query_retrieval_metrics_path}")
print(f"Scored {len(query_retrieval_metrics):,} retrieval queries")

skipped_contexts = pd.DataFrame(skipped_context_records)
if not skipped_contexts.empty:
    skipped_contexts_path = OUTPUT_DIR / "skipped_retrieval_contexts.tsv"
    skipped_contexts.to_csv(skipped_contexts_path, sep="	", index=False)
    print(f"Saved skipped retrieval contexts to {skipped_contexts_path}")

unresolved_pool_frame = pd.DataFrame(unresolved_pool_records)
if not unresolved_pool_frame.empty:
    unresolved_pool_path = OUTPUT_DIR / "unresolved_retrieval_pool_rows.tsv"
    unresolved_pool_frame.to_csv(unresolved_pool_path, sep="	", index=False)
    print(f"Saved unresolved retrieval pool rows to {unresolved_pool_path}")

stratum_retrieval_summary = (
    query_retrieval_metrics.groupby(
        ["dataset_a", "dataset_b", "direction", "query_dataset", "target_dataset", "cell_type", "time_key", "representation", "retrieval_variant"],
        as_index=False,
    )
    .agg(
        n_queries=("query_obs_id", "size"),
        n_unique_query_compounds=("query_pubchem_cid", "nunique"),
        mean_target_pool_size=("target_pool_size", "mean"),
        mean_shared_genes=("n_shared_genes", "mean"),
        mean_positive_candidates=("n_positive_candidates", "mean"),
        mean_normalized_best_positive_rank=("normalized_best_positive_rank", "mean"),
        mean_recall_at_1=("recall_at_1", "mean"),
        mean_auroc=("auroc", "mean"),
    )
    .sort_values(["dataset_a", "dataset_b", "direction", "cell_type", "time_key", "representation", "retrieval_variant"])
    .reset_index(drop=True)
)
stratum_retrieval_summary_path = OUTPUT_DIR / "line_time_retrieval_summary.tsv"
stratum_retrieval_summary.to_csv(stratum_retrieval_summary_path, sep="	", index=False)
print(f"Saved line-time retrieval summary to {stratum_retrieval_summary_path}")

pair_direction_retrieval_summary = (
    stratum_retrieval_summary.groupby(
        ["dataset_a", "dataset_b", "direction", "query_dataset", "target_dataset", "representation", "retrieval_variant"],
        as_index=False,
    )
    .agg(
        n_line_time_strata=("cell_type", "size"),
        n_queries=("n_queries", "sum"),
        mean_normalized_best_positive_rank=("mean_normalized_best_positive_rank", "mean"),
        mean_recall_at_1=("mean_recall_at_1", "mean"),
        mean_auroc=("mean_auroc", "mean"),
    )
    .sort_values(["dataset_a", "dataset_b", "direction", "representation", "retrieval_variant"])
    .reset_index(drop=True)
)
pair_direction_retrieval_summary_path = OUTPUT_DIR / "dataset_pair_direction_retrieval_summary.tsv"
pair_direction_retrieval_summary.to_csv(pair_direction_retrieval_summary_path, sep="	", index=False)
print(f"Saved pair-direction retrieval summary to {pair_direction_retrieval_summary_path}")

pair_retrieval_summary = (
    pair_direction_retrieval_summary.groupby(["dataset_a", "dataset_b", "representation", "retrieval_variant"], as_index=False)
    .agg(
        n_directions=("direction", "size"),
        mean_line_time_strata=("n_line_time_strata", "mean"),
        mean_queries=("n_queries", "mean"),
        mean_normalized_best_positive_rank=("mean_normalized_best_positive_rank", "mean"),
        mean_recall_at_1=("mean_recall_at_1", "mean"),
        mean_auroc=("mean_auroc", "mean"),
    )
    .sort_values(["dataset_a", "dataset_b", "representation", "retrieval_variant"])
    .reset_index(drop=True)
)
pair_retrieval_summary_path = OUTPUT_DIR / "dataset_pair_retrieval_summary.tsv"
pair_retrieval_summary.to_csv(pair_retrieval_summary_path, sep="	", index=False)
print(f"Saved symmetric dataset-pair retrieval summary to {pair_retrieval_summary_path}")

line_retrieval_summary = (
    stratum_retrieval_summary.groupby(["dataset_a", "dataset_b", "cell_type", "representation", "retrieval_variant"], as_index=False)
    .agg(
        n_time_strata=("time_key", "size"),
        n_queries=("n_queries", "sum"),
        mean_normalized_best_positive_rank=("mean_normalized_best_positive_rank", "mean"),
        mean_recall_at_1=("mean_recall_at_1", "mean"),
        mean_auroc=("mean_auroc", "mean"),
    )
    .sort_values(["dataset_a", "dataset_b", "cell_type", "representation", "retrieval_variant"])
    .reset_index(drop=True)
)
line_retrieval_summary_path = OUTPUT_DIR / "dataset_pair_line_retrieval_summary.tsv"
line_retrieval_summary.to_csv(line_retrieval_summary_path, sep="	", index=False)
print(f"Saved dataset-pair-line retrieval summary to {line_retrieval_summary_path}")

query_retrieval_metrics.head()


Saved query-level retrieval metrics to /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_group_rep_retrieval_metrics/query_retrieval_metrics.tsv
Scored 102,495 retrieval queries
Saved unresolved retrieval pool rows to /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_group_rep_retrieval_metrics/unresolved_retrieval_pool_rows.tsv
Saved line-time retrieval summary to /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_group_rep_retrieval_metrics/line_time_retrieval_summary.tsv
Saved pair-direction retrieval summary to /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_group_rep_retrieval_metrics/dataset_pair_direction_retrieval_summary.tsv
Saved symmetric dataset-pair retrieval summary to /ictstr01/groups/ml01/workspace/artur.szalata/code/op3_analysis/results/overlap_group_rep_retrieval_metrics/dataset_pair_retrieval_summary.tsv
Saved dataset-pair-line retrieval summary to

,dataset_a,dataset_b,direction,query_dataset,target_dataset,cell_type,time_key,representation,retrieval_variant,query_obs_id,...,n_target_side_unique_compounds,n_shared_genes,n_positive_candidates,best_positive_rank,normalized_best_positive_rank,recall_at_1,auroc,random_recall_at_1,random_normalized_best_positive_rank,random_auroc
0,l1000_phase1,l1000_phase2,A_to_B,l1000_phase1,l1000_phase2,CVCL_0023,24,signed_significance,compound_across_doses,CPC005_A549_24H_X1_B3_DUO52HI53LO_G05_GSK-3-in...,...,166,978,5,677.0,0.649378,0.0,0.648649,0.002592,0.5,0.5
1,l1000_phase1,l1000_phase2,A_to_B,l1000_phase1,l1000_phase2,CVCL_0023,24,signed_significance,dose_aware_compound,CPC005_A549_24H_X1_B3_DUO52HI53LO_G05_GSK-3-in...,...,166,978,5,677.0,0.649378,0.0,0.648649,0.002592,0.5,0.5
2,l1000_phase1,l1000_phase2,A_to_B,l1000_phase1,l1000_phase2,CVCL_0023,24,signed_significance,strict_matched_condition,CPC005_A549_24H_X1_B3_DUO52HI53LO_G05_GSK-3-in...,...,166,978,5,677.0,0.649378,0.0,0.648649,0.002592,0.5,0.5
3,l1000_phase1,l1000_phase2,A_to_B,l1000_phase1,l1000_phase2,CVCL_0023,24,signed_significance,compound_across_doses,CPC005_A549_24H_X2_B3_DUO52HI53LO_G05_GSK-3-in...,...,166,978,5,677.0,0.649378,0.0,0.648649,0.002592,0.5,0.5
4,l1000_phase1,l1000_phase2,A_to_B,l1000_phase1,l1000_phase2,CVCL_0023,24,signed_significance,dose_aware_compound,CPC005_A549_24H_X2_B3_DUO52HI53LO_G05_GSK-3-in...,...,166,978,5,677.0,0.649378,0.0,0.648649,0.002592,0.5,0.5


**Primary Retrieval Task: Compound Retrieval Across Doses**

The main score is normalized best-positive rank. Random baseline is `0.5`.

This section now keeps both:
- direction-specific summaries (`A -> B` and `B -> A`) without averaging across directions
- symmetric dataset-pair summaries that average the two directions


In [59]:
primary_direction_summary = pair_direction_retrieval_summary.loc[
    pair_direction_retrieval_summary["retrieval_variant"] == "compound_across_doses"
].copy()
primary_pair_summary = pair_retrieval_summary.loc[
    pair_retrieval_summary["retrieval_variant"] == "compound_across_doses"
].copy()

primary_direction_summary["dataset_a"] = primary_direction_summary["dataset_a"].map(pretty_label)
primary_direction_summary["dataset_b"] = primary_direction_summary["dataset_b"].map(pretty_label)
primary_direction_summary["query_dataset"] = primary_direction_summary["query_dataset"].map(pretty_label)
primary_direction_summary["target_dataset"] = primary_direction_summary["target_dataset"].map(pretty_label)
primary_pair_summary["dataset_a"] = primary_pair_summary["dataset_a"].map(pretty_label)
primary_pair_summary["dataset_b"] = primary_pair_summary["dataset_b"].map(pretty_label)

for representation_name, representation_label in REPRESENTATION_LABELS.items():
    print(representation_label)

    direction_frame = primary_direction_summary.loc[
        primary_direction_summary["representation"] == representation_name
    ].copy()
    print("Direction-specific summary")
    display(direction_frame)
    display(direction_frame.pivot(index="query_dataset", columns="target_dataset", values="mean_normalized_best_positive_rank"))
    display(direction_frame.pivot(index="query_dataset", columns="target_dataset", values="mean_recall_at_1"))
    display(direction_frame.pivot(index="query_dataset", columns="target_dataset", values="mean_auroc"))

    representation_frame = primary_pair_summary.loc[
        primary_pair_summary["representation"] == representation_name
    ].copy()
    print("Symmetric pair summary")
    display(representation_frame)
    display(representation_frame.pivot(index="dataset_a", columns="dataset_b", values="mean_normalized_best_positive_rank"))
    display(representation_frame.pivot(index="dataset_a", columns="dataset_b", values="mean_recall_at_1"))
    display(representation_frame.pivot(index="dataset_a", columns="dataset_b", values="mean_auroc"))


Signed significance
Direction-specific summary


,dataset_a,dataset_b,direction,query_dataset,target_dataset,representation,retrieval_variant,n_line_time_strata,n_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
6,L1000 Phase I,L1000 Phase II,A_to_B,L1000 Phase I,L1000 Phase II,signed_significance,compound_across_doses,4,1878,0.776438,0.184179,0.642141
15,L1000 Phase I,L1000 Phase II,B_to_A,L1000 Phase II,L1000 Phase I,signed_significance,compound_across_doses,4,2301,0.698274,0.134078,0.629851
24,L1000 Phase I,sci-Plex,A_to_B,L1000 Phase I,sci-Plex,signed_significance,compound_across_doses,2,1413,0.767685,0.078588,0.594697
33,L1000 Phase I,sci-Plex,B_to_A,sci-Plex,L1000 Phase I,signed_significance,compound_across_doses,2,765,0.689923,0.014541,0.574703
42,L1000 Phase I,Tahoe-100M,A_to_B,L1000 Phase I,Tahoe-100M,signed_significance,compound_across_doses,3,529,0.711537,0.097864,0.580924
51,L1000 Phase I,Tahoe-100M,B_to_A,Tahoe-100M,L1000 Phase I,signed_significance,compound_across_doses,3,240,0.603472,0.047229,0.526851
60,L1000 Phase II,sci-Plex,A_to_B,L1000 Phase II,sci-Plex,signed_significance,compound_across_doses,2,1820,0.831615,0.106014,0.653960
69,L1000 Phase II,sci-Plex,B_to_A,sci-Plex,L1000 Phase II,signed_significance,compound_across_doses,2,771,0.750009,0.023212,0.575490
78,L1000 Phase II,Tahoe-100M,A_to_B,L1000 Phase II,Tahoe-100M,signed_significance,compound_across_doses,2,1445,0.763567,0.058892,0.576832
87,L1000 Phase II,Tahoe-100M,B_to_A,Tahoe-100M,L1000 Phase II,signed_significance,compound_across_doses,2,456,0.717889,0.010046,0.534852


target_dataset,L1000 Phase I,L1000 Phase II,Tahoe-100M,sci-Plex
query_dataset,,,,
L1000 Phase I,NaN,0.776438,0.711537,0.767685
L1000 Phase II,0.698274,NaN,0.763567,0.831615
Tahoe-100M,0.603472,0.717889,NaN,0.753362
sci-Plex,0.689923,0.750009,0.709888,NaN


target_dataset,L1000 Phase I,L1000 Phase II,Tahoe-100M,sci-Plex
query_dataset,,,,
L1000 Phase I,NaN,0.184179,0.097864,0.078588
L1000 Phase II,0.134078,NaN,0.058892,0.106014
Tahoe-100M,0.047229,0.010046,NaN,0.152941
sci-Plex,0.014541,0.023212,0.057971,NaN


target_dataset,L1000 Phase I,L1000 Phase II,Tahoe-100M,sci-Plex
query_dataset,,,,
L1000 Phase I,NaN,0.642141,0.580924,0.594697
L1000 Phase II,0.629851,NaN,0.576832,0.653960
Tahoe-100M,0.526851,0.534852,NaN,0.571723
sci-Plex,0.574703,0.575490,0.525554,NaN


...

Symmetric pair summary


,dataset_a,dataset_b,representation,retrieval_variant,n_directions,mean_line_time_strata,mean_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
0,L1000 Phase I,L1000 Phase II,logFC,compound_across_doses,2,4.0,2089.5,0.724541,0.183835,0.644635
9,L1000 Phase I,sci-Plex,logFC,compound_across_doses,2,2.0,1089.0,0.662043,0.027102,0.551292
18,L1000 Phase I,Tahoe-100M,logFC,compound_across_doses,2,3.0,384.5,0.639502,0.077087,0.525544
27,L1000 Phase II,sci-Plex,logFC,compound_across_doses,2,2.0,1295.5,0.773143,0.038799,0.596161
36,L1000 Phase II,Tahoe-100M,logFC,compound_across_doses,2,2.0,950.5,0.683610,0.044096,0.516151
45,sci-Plex,Tahoe-100M,logFC,compound_across_doses,2,1.0,111.5,0.712734,0.062489,0.549358


dataset_b,L1000 Phase II,Tahoe-100M,sci-Plex
dataset_a,,,
L1000 Phase I,0.724541,0.639502,0.662043
L1000 Phase II,NaN,0.683610,0.773143
sci-Plex,NaN,0.712734,NaN


dataset_b,L1000 Phase II,Tahoe-100M,sci-Plex
dataset_a,,,
L1000 Phase I,0.183835,0.077087,0.027102
L1000 Phase II,NaN,0.044096,0.038799
sci-Plex,NaN,0.062489,NaN


dataset_b,L1000 Phase II,Tahoe-100M,sci-Plex
dataset_a,,,
L1000 Phase I,0.644635,0.525544,0.551292
L1000 Phase II,NaN,0.516151,0.596161
sci-Plex,NaN,0.549358,NaN


**Sensitivity Variants**

These tables summarize the stricter retrieval variants:
- dose-aware compound retrieval
- strict matched-condition retrieval

For each variant, the notebook keeps both direction-specific and symmetric pair summaries.


In [60]:
sensitivity_direction_summary = pair_direction_retrieval_summary.loc[
    pair_direction_retrieval_summary["retrieval_variant"] != "compound_across_doses"
].copy()
sensitivity_pair_summary = pair_retrieval_summary.loc[
    pair_retrieval_summary["retrieval_variant"] != "compound_across_doses"
].copy()

sensitivity_direction_summary["dataset_a"] = sensitivity_direction_summary["dataset_a"].map(pretty_label)
sensitivity_direction_summary["dataset_b"] = sensitivity_direction_summary["dataset_b"].map(pretty_label)
sensitivity_direction_summary["query_dataset"] = sensitivity_direction_summary["query_dataset"].map(pretty_label)
sensitivity_direction_summary["target_dataset"] = sensitivity_direction_summary["target_dataset"].map(pretty_label)
sensitivity_pair_summary["dataset_a"] = sensitivity_pair_summary["dataset_a"].map(pretty_label)
sensitivity_pair_summary["dataset_b"] = sensitivity_pair_summary["dataset_b"].map(pretty_label)

for retrieval_variant, retrieval_label in RETRIEVAL_VARIANTS.items():
    if retrieval_variant == "compound_across_doses":
        continue
    print(retrieval_label)
    variant_direction_frame = sensitivity_direction_summary.loc[
        sensitivity_direction_summary["retrieval_variant"] == retrieval_variant
    ].copy()
    variant_pair_frame = sensitivity_pair_summary.loc[
        sensitivity_pair_summary["retrieval_variant"] == retrieval_variant
    ].copy()
    for representation_name, representation_label in REPRESENTATION_LABELS.items():
        print(representation_label)
        direction_frame = variant_direction_frame.loc[
            variant_direction_frame["representation"] == representation_name
        ].copy()
        print("Direction-specific summary")
        display(direction_frame)
        pair_frame = variant_pair_frame.loc[
            variant_pair_frame["representation"] == representation_name
        ].copy()
        print("Symmetric pair summary")
        display(pair_frame)


Dose-aware compound retrieval
Signed significance
Direction-specific summary


,dataset_a,dataset_b,direction,query_dataset,target_dataset,representation,retrieval_variant,n_line_time_strata,n_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
7,L1000 Phase I,L1000 Phase II,A_to_B,L1000 Phase I,L1000 Phase II,signed_significance,dose_aware_compound,4,1878,0.727601,0.182281,0.662907
16,L1000 Phase I,L1000 Phase II,B_to_A,L1000 Phase II,L1000 Phase I,signed_significance,dose_aware_compound,4,2301,0.691434,0.122827,0.640845
25,L1000 Phase I,sci-Plex,A_to_B,L1000 Phase I,sci-Plex,signed_significance,dose_aware_compound,2,1413,0.721455,0.071844,0.622291
34,L1000 Phase I,sci-Plex,B_to_A,sci-Plex,L1000 Phase I,signed_significance,dose_aware_compound,2,765,0.652404,0.008949,0.593673
43,L1000 Phase I,Tahoe-100M,A_to_B,L1000 Phase I,Tahoe-100M,signed_significance,dose_aware_compound,3,529,0.627448,0.048219,0.594729
52,L1000 Phase I,Tahoe-100M,B_to_A,Tahoe-100M,L1000 Phase I,signed_significance,dose_aware_compound,3,240,0.555602,0.023845,0.520625
61,L1000 Phase II,sci-Plex,A_to_B,L1000 Phase II,sci-Plex,signed_significance,dose_aware_compound,2,1820,0.781493,0.096123,0.679319
70,L1000 Phase II,sci-Plex,B_to_A,sci-Plex,L1000 Phase II,signed_significance,dose_aware_compound,2,771,0.699360,0.013563,0.592662
79,L1000 Phase II,Tahoe-100M,A_to_B,L1000 Phase II,Tahoe-100M,signed_significance,dose_aware_compound,2,1445,0.694635,0.036678,0.611592
88,L1000 Phase II,Tahoe-100M,B_to_A,Tahoe-100M,L1000 Phase II,signed_significance,dose_aware_compound,2,456,0.622112,0.010046,0.532603


Symmetric pair summary


,dataset_a,dataset_b,representation,retrieval_variant,n_directions,mean_line_time_strata,mean_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
7,L1000 Phase I,L1000 Phase II,signed_significance,dose_aware_compound,2,4.0,2089.5,0.709518,0.152554,0.651876
16,L1000 Phase I,sci-Plex,signed_significance,dose_aware_compound,2,2.0,1089.0,0.686930,0.040396,0.607982
25,L1000 Phase I,Tahoe-100M,signed_significance,dose_aware_compound,2,3.0,384.5,0.591525,0.036032,0.557677
34,L1000 Phase II,sci-Plex,signed_significance,dose_aware_compound,2,2.0,1295.5,0.740427,0.054843,0.635990
43,L1000 Phase II,Tahoe-100M,signed_significance,dose_aware_compound,2,2.0,950.5,0.658374,0.023362,0.572097
52,sci-Plex,Tahoe-100M,signed_significance,dose_aware_compound,2,1.0,111.5,0.629297,0.073316,0.555246


Moderated t
Direction-specific summary


...

,dataset_a,dataset_b,representation,retrieval_variant,n_directions,mean_line_time_strata,mean_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
5,L1000 Phase I,L1000 Phase II,moderated_t,strict_matched_condition,2,4.0,1796.5,0.781790,0.166417,0.778551
14,L1000 Phase I,sci-Plex,moderated_t,strict_matched_condition,2,2.0,948.5,0.679522,0.037170,0.678518
23,L1000 Phase I,Tahoe-100M,moderated_t,strict_matched_condition,2,3.0,346.0,0.601241,0.035987,0.593519
32,L1000 Phase II,sci-Plex,moderated_t,strict_matched_condition,2,2.0,1161.5,0.702943,0.038920,0.702178
41,L1000 Phase II,Tahoe-100M,moderated_t,strict_matched_condition,2,2.0,877.5,0.588637,0.009561,0.586951
50,sci-Plex,Tahoe-100M,moderated_t,strict_matched_condition,2,1.0,111.5,0.615709,0.024893,0.613162


logFC
Direction-specific summary


,dataset_a,dataset_b,direction,query_dataset,target_dataset,representation,retrieval_variant,n_line_time_strata,n_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
2,L1000 Phase I,L1000 Phase II,A_to_B,L1000 Phase I,L1000 Phase II,logFC,strict_matched_condition,4,1853,0.771228,0.167962,0.769208
11,L1000 Phase I,L1000 Phase II,B_to_A,L1000 Phase II,L1000 Phase I,logFC,strict_matched_condition,4,1740,0.617534,0.142138,0.612569
20,L1000 Phase I,sci-Plex,A_to_B,L1000 Phase I,sci-Plex,logFC,strict_matched_condition,2,1318,0.623115,0.022201,0.622415
29,L1000 Phase I,sci-Plex,B_to_A,sci-Plex,L1000 Phase I,logFC,strict_matched_condition,2,579,0.541672,0.010091,0.540569
38,L1000 Phase I,Tahoe-100M,A_to_B,L1000 Phase I,Tahoe-100M,logFC,strict_matched_condition,3,508,0.584523,0.051187,0.578345
47,L1000 Phase I,Tahoe-100M,B_to_A,Tahoe-100M,L1000 Phase I,logFC,strict_matched_condition,3,184,0.536726,0.023301,0.529234
56,L1000 Phase II,sci-Plex,A_to_B,L1000 Phase II,sci-Plex,logFC,strict_matched_condition,2,1570,0.625757,0.013404,0.625075
65,L1000 Phase II,sci-Plex,B_to_A,sci-Plex,L1000 Phase II,logFC,strict_matched_condition,2,753,0.661097,0.025768,0.660348
74,L1000 Phase II,Tahoe-100M,A_to_B,L1000 Phase II,Tahoe-100M,logFC,strict_matched_condition,2,1299,0.511036,0.004687,0.509896
83,L1000 Phase II,Tahoe-100M,B_to_A,Tahoe-100M,L1000 Phase II,logFC,strict_matched_condition,2,456,0.523427,0.005362,0.522606


Symmetric pair summary


,dataset_a,dataset_b,representation,retrieval_variant,n_directions,mean_line_time_strata,mean_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
2,L1000 Phase I,L1000 Phase II,logFC,strict_matched_condition,2,4.0,1796.5,0.694381,0.155050,0.690889
11,L1000 Phase I,sci-Plex,logFC,strict_matched_condition,2,2.0,948.5,0.582394,0.016146,0.581492
20,L1000 Phase I,Tahoe-100M,logFC,strict_matched_condition,2,3.0,346.0,0.560624,0.037244,0.553789
29,L1000 Phase II,sci-Plex,logFC,strict_matched_condition,2,2.0,1161.5,0.643427,0.019586,0.642712
38,L1000 Phase II,Tahoe-100M,logFC,strict_matched_condition,2,2.0,877.5,0.517232,0.005025,0.516251
47,sci-Plex,Tahoe-100M,logFC,strict_matched_condition,2,1.0,111.5,0.562807,0.034868,0.560593


**Line-Level Primary Retrieval Summary**

These line-level summaries average across matched `time_key` strata and both directions for the primary across-dose compound retrieval task.


In [61]:
primary_line_summary = line_retrieval_summary.loc[
    line_retrieval_summary["retrieval_variant"] == "compound_across_doses"
].copy()
primary_line_summary["dataset_a"] = primary_line_summary["dataset_a"].map(pretty_label)
primary_line_summary["dataset_b"] = primary_line_summary["dataset_b"].map(pretty_label)
display(primary_line_summary)


,dataset_a,dataset_b,cell_type,representation,retrieval_variant,n_time_strata,n_queries,mean_normalized_best_positive_rank,mean_recall_at_1,mean_auroc
0,L1000 Phase I,L1000 Phase II,CVCL_0023,logFC,compound_across_doses,4,2742,0.712468,0.254897,0.662777
3,L1000 Phase I,L1000 Phase II,CVCL_0023,moderated_t,compound_across_doses,4,2742,0.803604,0.255036,0.746748
6,L1000 Phase I,L1000 Phase II,CVCL_0023,signed_significance,compound_across_doses,4,2742,0.731703,0.195783,0.657686
9,L1000 Phase I,L1000 Phase II,CVCL_0031,logFC,compound_across_doses,2,1270,0.737643,0.144615,0.605594
12,L1000 Phase I,L1000 Phase II,CVCL_0031,moderated_t,compound_across_doses,2,1270,0.839294,0.159982,0.686920
15,L1000 Phase I,L1000 Phase II,CVCL_0031,signed_significance,compound_across_doses,2,1270,0.807831,0.150855,0.656201
18,L1000 Phase I,L1000 Phase II,CVCL_0320,logFC,compound_across_doses,2,167,0.735585,0.080933,0.647392
21,L1000 Phase I,L1000 Phase II,CVCL_0320,moderated_t,compound_across_doses,2,167,0.739855,0.052632,0.654738
24,L1000 Phase I,L1000 Phase II,CVCL_0320,signed_significance,compound_across_doses,2,167,0.678187,0.094091,0.572412
27,L1000 Phase I,sci-Plex,CVCL_0023,logFC,compound_across_doses,2,837,0.631111,0.017850,0.519591


Optional cleanup:

```python
close_all_line_sources()
```
